# Uni-D 후보 모델 실험

이 노트북은 실험 설계와 확인용입니다. **학습과 추론은 여기서 실행하지 않습니다.** 오래 걸리는 학습은 VS Code 터미널의 tmux 안에서 `python train.py`로 돌립니다. 연결이 끊기거나 노트북 덮개를 닫아도 그때의 학습은 유지됩니다.

코드의 기준은 `model.py`입니다. `train.py`와 `test.py`가 그 함수로 학습하고 예측합니다. 운영진은 `train.py`를 실행하지 않고 아래 한 줄만 돌립니다.

```bash
python test.py --input <CSV> --image_root <이미지 폴더> --output <저장 경로>
```

선택 기준은 Train 실험 단위 교차검증 Macro F1입니다. Validation 점수는 후보를 고른 뒤에 한 번만 확인하고, 그 숫자로 모델을 고르지 않습니다. Test는 추론에만 씁니다.

## 후보

| 순위 | 모델 | 종류 | 입력 |
|---|---|---|---|
| 1 | LightGBM, XGBoost, CatBoost, Extra Trees, HistGradientBoosting | 표 | 확장 피처 |
| 1 | cumulative | 표 | “k보다 큰가” 이항 5개 |
| 2 | EfficientNet-B0, ResNet18, MobileNetV3-Small | 딥러닝 | 현재 열화상 + 센서 피처 |
| 2 | smallcnn | 딥러닝 | 열화상 전용 작은 합성곱망 + 센서 피처 |
| 2 | convnext_tiny, swin_t, deit_small, dinov2_vits14, maxvit_tiny, mobilevit_s | 딥러닝 | 현재 열화상 + 센서 피처 |
| 2 | resnet18_gapgmp | 딥러닝 | ResNet18 평균 풀링 + 최대 풀링 |
| 2 | swin_t_time | 딥러닝 | 현재, 30초 전, 120초 전 열화상 |
| 2 | cnn1d | 딥러닝 | 과거 300초 센서. 이미지는 읽지 않음 |
| 3 | blend.py | 조합 | 저장된 확률의 평균, 홀드아웃 비중, 스택 |
| 3 | `--head coral` | 조합 | 딥러닝 출력을 순서형 5개로 |

확장 피처는 센서 5개, 제공 CSV에 있는 가스·`thermal_t`·`atmp_t` 등, 열화상 밝기·고온 영역, 30·60·120·300초의 과거 평균·최댓값·변화·온도 기울기, 실험 시작 시각 대비 변화입니다. `indu_t.1`은 넣지 않습니다.

확률은 평균하고, 확정 라벨의 다수결은 쓰지 않습니다. 조합 비율은 학습 스크립트 안에서 Train 교차검증으로만 정해지며 기본 앙상블은 같은 비중입니다.

## 공통 규칙

| 항목 | 방법 |
|---|---|
| 분할 | Train 실험을 GroupKFold. 같은 실험이 학습과 검증에 동시에 들어가지 않음 |
| 표 모델 | 5분할 Macro F1. 4단계·5단계 F1을 같이 기록 |
| 딥러닝 | 기본 1개 홀드아웃으로 epoch을 고른 뒤 Train 전체로 다시 학습 |
| Validation | 선택 후 한 번만 확인 |
| 시계열 | 같은 실험, 현재 시각 이전과 현재. 단위는 초 |
| 전처리 | 중앙값·표준화는 그 모델을 학습하는 행에서만 계산해 가중치에 저장 |
| 추론 | `eval`, 고정 시드, 고정 리사이즈. 무작위 뒤집기는 학습에만 |
| 이미지 | 샘플마다 계산. 미리 만들어 둔 피처 파일을 `test.py`가 읽지 않음 |

사전학습 가중치는 대회 시작 전에 공개된 torchvision ImageNet-1K V1입니다. 이 대회 데이터로 학습된 외부 가중치와 외부 추론 API는 쓰지 않습니다.

In [ ]:
import model as M

print('표 모델:', ', '.join(M.TABULAR_MODELS))
print('딥러닝:', ', '.join(M.DEEP_MODELS))
print('조합:', ', '.join(M.COMBINATION_MODELS))
print('윈도우(초):', M.WINDOWS)
print('시드:', M.SEED)
print('이 셀은 모델을 학습하지 않습니다. 학습은 tmux의 train.py에서만 실행합니다.')


## tmux에서 실행할 명령

세션을 만든 뒤 화면이 바뀌면 나머지를 입력합니다. 로그는 `train.log`에 남고, 끝나면 교차검증 Macro F1과 Validation Macro F1이 출력됩니다.

```bash
tmux new -s train-<내이름>
source ~/venv/bin/activate
cd ~/<내이름>/Uni-Dthon-Team-16
pip install lightgbm xgboost catboost
D=~/data/battery_data_light
python -u train.py --data_dir $D --compare tabular --model_out weights/model.joblib 2>&1 | tee train.log
```

`Ctrl+B` 다음 `D`로 세션에서 나옵니다. 다시 들어가려면 `tmux attach -t train-<내이름>`. 세션 목록은 `tmux ls`입니다.

아직 하지 않은 실험은 아래 순서입니다. 한 번에 하나만 실행합니다. gru_efficientnet은 제외합니다. 이미지 피처와 torch 스레드는 기본 2개입니다. 딥러닝을 돌릴 때는 다른 터미널에서 `watch -n 2 nvidia-smi`로 GPU를 확인합니다. 끝낼 때는 `Ctrl+C`입니다.

학습 없이 바로 할 수 있는 조합입니다. 확률을 평균하고, 확정 라벨의 다수결은 쓰지 않습니다.

```bash
python blend.py --models weights/efficientnet.joblib weights/resnet18.joblib --input $D/test/test.csv --image_root $D/test --output weights/submission_eff_resnet.csv --model_out weights/eff_resnet.joblib
python blend.py --models weights/lightgbm.joblib weights/xgboost.joblib weights/catboost.joblib weights/extratrees.joblib weights/histgb.joblib weights/cumulative.joblib --input $D/test/test.csv --image_root $D/test --output weights/submission_ensemble.csv --model_out weights/ensemble.joblib
```

이어서 학습은 하나씩 실행합니다. 첫 모델은 Swin-Tiny 결합망의 CORAL 헤드이고, 다음은 같은 모델의 교차엔트로피 헤드와 ConvNeXt-Tiny입니다.

```bash
python -u train.py --data_dir $D --model swin_t --head coral --model_out weights/swin_t_coral.joblib 2>&1 | tee train_swin_coral.log
python -u train.py --data_dir $D --model swin_t --model_out weights/swin_t.joblib 2>&1 | tee train_swin.log
python -u train.py --data_dir $D --model convnext_tiny --model_out weights/convnext_tiny.joblib 2>&1 | tee train_convnext.log
python -u train.py --data_dir $D --model resnet18_gapgmp --model_out weights/resnet18_gapgmp.joblib 2>&1 | tee train_resnet_gapgmp.log
python -u train.py --data_dir $D --model cnn1d --model_out weights/cnn1d.joblib 2>&1 | tee train_cnn1d.log
```

Swin 한 장의 4단계 F1과 5단계 F1이 모두 LightGBM(0.8835, 0.8811)보다 낮으면 세 시점과 나머지 이미지 모델을 하나씩 실행합니다.

```bash
python -u train.py --data_dir $D --model swin_t_time --model_out weights/swin_t_time.joblib 2>&1 | tee train_swin_time.log
python -u train.py --data_dir $D --model deit_small --model_out weights/deit_small.joblib 2>&1 | tee train_deit.log
python -u train.py --data_dir $D --model dinov2_vits14 --model_out weights/dinov2_vits14.joblib 2>&1 | tee train_dinov2.log
python -u train.py --data_dir $D --model maxvit_tiny --model_out weights/maxvit_tiny.joblib 2>&1 | tee train_maxvit.log
python -u train.py --data_dir $D --model mobilevit_s --model_out weights/mobilevit_s.joblib 2>&1 | tee train_mobilevit.log
```

단일 모델 교차검증이 LightGBM 0.8913을 넘으면 `--seed 43`, `--seed 44`로 같은 모델을 학습하고 세 파일의 확률을 평균합니다. 비중과 스택은 홀드아웃 확률 `*_oof.npz`가 생긴 뒤에만 실행합니다. 표 모델의 홀드아웃 확률은 기존 가중치를 유지한 채 다음 명령으로 만듭니다.

```bash
python train.py --data_dir $D --model lightgbm --oof_only --model_out weights/lightgbm.joblib
python blend.py --fit-weights --models weights/swin_t.joblib weights/lightgbm.joblib --input $D/test/test.csv --image_root $D/test --output weights/submission_swin_lgb.csv --model_out weights/swin_lgb.joblib
python blend.py --models weights/swin_t.joblib weights/swin_t_coral.joblib --input $D/test/test.csv --image_root $D/test --output weights/submission_swin_coral.csv --model_out weights/swin_coral.joblib
python blend.py --stack --models weights/swin_t.joblib weights/lightgbm.joblib --input $D/test/test.csv --image_root $D/test --output weights/submission_stack.csv --model_out weights/stack.joblib
```

각 후보는 `weights/<모델>.joblib`에 남습니다. 리더보드에 낸 가중치는 지우지 않습니다. 최종으로 고른 파일만 `weights/model.joblib`로 두면 `test.py` 기본 경로와 같습니다.

```bash
python test.py --input $D/test/test.csv --image_root $D/test --output submission.csv
```

제출 전에 `check_submission.py`로 재현, 같은 입력 재실행, 행 순서, 실험 분리, 미래 행 제거가 모두 통과하는지 확인합니다. 최종 파일은 이 `test.py`가 만든 CSV입니다.

## 로그에서 볼 표

`train.py`가 아래 열로 한 줄씩 출력합니다. `cv_f1`이 가장 높은 모델을 저장하고, `val_f1`은 참고값입니다.

| 열 | 의미 |
|---|---|
| cv_f1 | Train 실험 교차검증 Macro F1. 선택 기준 |
| cv_std | 분할 간 표준편차. 딥러닝 1분할이면 `-` |
| f1_4, f1_5 | 그 교차검증의 4단계·5단계 F1 |
| val_f1 | 학습에 넣지 않은 Validation Macro F1 |

같은 평균이면 목록에서 앞에 있는 모델을 남깁니다. 4·5단계 F1이 무너진 후보는 평균이 조금 높아도 최종 파일로 올리지 않습니다.